# Raw data grab: every upstream source of every fire, one folder per fire

Downloads all channels of upstream data behind the wildfire spread dataset
([Javyn123/wildfire-spread-2014-2025](https://huggingface.co/datasets/Javyn123/wildfire-spread-2014-2025)) into a
Hugging Face dataset repo you choose. Each fire ends up with one folder:

```
fires/<fire_id>/
  LINKS.md, links.json    every channel of this fire: status, links to its files here, and the upstream URLs
  history/  perimeters/  firms/  gridmet/  ics209/  suppression/  tessera/  terrain/
  landfire/  goes/  hrrr/  osm/
```

Each channel holds that source's own records, values and grid, cut to the fire's box and dates. The code is
`pipeline/raw_grab.py` on the `data-grab` branch of
[WildfireRepo1](https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1).

**Before you run**
- Settings → Internet: **on**.
- Add-ons → Secrets:
  - `HF_TOKEN`: a Hugging Face token with write access
  - `FIRMS_MAP_KEY`: free at https://firms.modaps.eosdis.nasa.gov/api/map_key/
  - `LANDFIRE_EMAIL`: the LANDFIRE service needs a contact email

**How it runs**
- *Run all* goes through every channel, cheapest first. For each channel it fetches every fire, uploads them in
  batches of about 2 GB, and deletes each batch locally once it is uploaded.
- It works channel by channel, not fire by fire, so each big shared download happens once (an OSM snapshot is
  2–3 GB, and one GOES or HRRR file covers many fires). The repo still ends up organised by fire.
- At the end of the session it rebuilds every fire's `LINKS.md` / `links.json`.
- A full pull takes several 12-hour sessions, mostly `goes`, `hrrr` and `osm`. Just run the notebook again: fires
  already in the repo's manifests are skipped, and each session picks up where the last one stopped.

In [ ]:
# ---- settings
HF_REPO = "your-name/wildfire-raw-2014-2025"   # target dataset repo (created if missing)
PRIVATE = True                                 # only used when the repo is created
FIRES = "built"                                # built (1,841 fires with a built folder) | keep | all
WORK = "/tmp/raw"                              # scratch: staging, batches, one-time downloads (not /kaggle/working)
MAX_HOURS = 11.5                               # whole session, under Kaggle's 12 h limit
LINKS_RESERVE_H = 0.25                         # time kept for the per-fire links at the end
BATCH_GB = 2.0

CHANNELS = ["history", "perimeters", "firms", "gridmet", "ics209", "suppression", "tessera", "terrain",
            "landfire", "goes", "hrrr", "osm"]
RUN = CHANNELS                                 # or a subset, e.g. ["firms", "goes"]

In [ ]:
# ---- the project code, data-grab branch
!git clone -q --depth 1 -b data-grab https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1.git 2>/dev/null || (cd WildfireRepo1 && git pull -q)
!cd WildfireRepo1 && git log -1 --format='%h %s'

In [ ]:
# ---- packages (Kaggle has numpy, pandas, geopandas, rasterio, xarray, netCDF4)
!pip install -q pyogrio pygrib "huggingface_hub>=0.24"
import shutil
if "osm" in RUN and not shutil.which("osmium"):
    !apt-get -qq update && apt-get -qq install -y osmium-tool aria2 > /dev/null

In [ ]:
# ---- secrets -> environment (never printed)
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("HF_TOKEN", "FIRMS_MAP_KEY", "LANDFIRE_EMAIL"):
        try:
            os.environ[_k] = _s.get_secret(_k)
        except Exception:
            pass
except ImportError:
    pass                                       # not on Kaggle: use the existing environment
print({k: ("set" if os.environ.get(k) else "missing") for k in ("HF_TOKEN", "FIRMS_MAP_KEY", "LANDFIRE_EMAIL")})
if not os.environ.get("FIRMS_MAP_KEY") and "firms" in RUN:
    print("no FIRMS_MAP_KEY: skipping firms"); RUN = [c for c in RUN if c != "firms"]
if not os.environ.get("LANDFIRE_EMAIL") and "landfire" in RUN:
    print("no LANDFIRE_EMAIL: skipping landfire"); RUN = [c for c in RUN if c != "landfire"]

In [ ]:
import subprocess, sys, time

SCRIPT = "WildfireRepo1/pipeline/raw_grab.py"

def _run(args):
    p = subprocess.Popen([sys.executable, "-u", SCRIPT, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def _opts(opts):
    out = []
    for k, v in opts.items():
        flag = "--" + k.replace("_", "-")
        if v is True:
            out.append(flag)
        elif isinstance(v, (list, tuple)):
            out += [flag, *map(str, v)]
        elif v not in (None, False):
            out += [flag, str(v)]
    return out

def grab(channel, **opts):
    """One channel for every fire (resumes from the repo's manifests). opts: ids=[...], limit=N, dry_run=True, max_hours=..."""
    args = ["--channel", channel, "--hf-repo", HF_REPO, "--work", WORK, "--fires", opts.pop("fires", FIRES),
            "--batch-gb", str(opts.pop("batch_gb", BATCH_GB))] + (["--private"] if PRIVATE else [])
    return _run(args + _opts(opts))

def links(**opts):
    """Rebuild fires/<id>/LINKS.md + links.json and catalog/raw_index.csv from the manifests."""
    return _run(["--links", "--hf-repo", HF_REPO, "--work", WORK, "--fires", opts.pop("fires", FIRES)] + _opts(opts))

def status():
    return _run(["--status", "--hf-repo", HF_REPO])

Optional check first: every channel for one fire, nothing uploaded. Camp Fire 2018 is `F2003403265`.

In [ ]:
# for ch in RUN:
#     grab(ch, ids=["F2003403265"], dry_run=True)
# !find /tmp/raw/batch -type f | sort | head -40

## Download all channels

In [ ]:
t0 = time.time()
for ch in RUN:
    left = MAX_HOURS - LINKS_RESERVE_H - (time.time() - t0) / 3600
    if left < 0.25:
        print(f"session time used up before {ch}: run the notebook again to continue")
        break
    print(f"\n========== {ch} ({left:.1f} h left) ==========")
    grab(ch, max_hours=round(left, 2))

## Per-fire links, then progress

In [ ]:
links()
status()